# Data Cleaning & Validation Pipeline

---

## 📌 Overview
Data cleaning and validation is a critical bridge between initial data exploration and business analysis. Raw operational data often contains inconsistencies, missing values, incorrect formatting, and duplicate entries. 

This notebook processes the raw datasets extracted during ingestion, applying programmatic transformations in Python to produce clean, reliable, and standardized data ready for SQL analysis and reporting.



In [2]:
import os
import urllib.parse
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sqlalchemy import create_engine

# 1. Setup & Connection
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME")

ENCODED_PASSWORD = urllib.parse.quote_plus(DB_PASSWORD)
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{ENCODED_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("✅ Connected to MySQL for Data Cleaning!")

✅ Connected to MySQL for Data Cleaning!


### 2.1 Clean Customers (`olist_customers`)
* Trim leading/trailing whitespace from text fields.
* Convert `customer_city` to Title Case and `customer_state` to Uppercase.

In [3]:
# 1. Load Customers Table
df_customers = pd.read_sql("SELECT * FROM olist_customers", con=engine)

# 2. Clean Text Formatting (Trim whitespace, Title Case for City, Upper Case for State)
df_customers["customer_city"] = df_customers["customer_city"].str.strip().str.title()
df_customers["customer_state"] = df_customers["customer_state"].str.strip().str.upper()

# 3. Verification Printout
print(f"✅ Customers Cleaned!")
print(f"• Total Rows: {len(df_customers):,}")
print(f"• Null Values: {df_customers.isnull().sum().sum()}")
print(f"• Sample City Output: {df_customers['customer_city'].head(3).tolist()}")
print(f"• Sample State Output: {df_customers['customer_state'].head(3).tolist()}")

✅ Customers Cleaned!
• Total Rows: 99,441
• Null Values: 0
• Sample City Output: ['Osasco', 'Itapecerica', 'Nova Venecia']
• Sample State Output: ['SP', 'MG', 'ES']


### 2.2 Clean Orders (`olist_orders`)
* Convert 5 date string columns into proper Pandas `datetime` objects.
* Strip whitespace and lowercase `order_status` strings.

In [4]:
df_orders = pd.read_sql("SELECT * FROM olist_orders", con=engine)

# Convert string columns to datetime
date_cols_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols_orders:
    df_orders[col] = pd.to_datetime(df_orders[col])

# Standardize status strings
df_orders["order_status"] = df_orders["order_status"].str.strip().str.lower()

print(f"✅ Orders Cleaned! ({len(df_orders):,} rows)")
print(f"• Sample Datetime Type Check: {type(df_orders['order_purchase_timestamp'].iloc[0])}")

✅ Orders Cleaned! (99,441 rows)
• Sample Datetime Type Check: <class 'pandas._libs.tslibs.timestamps.Timestamp'>


### 2.3 Clean Products (`olist_products`)
* Left join with `product_category_translation` to bring in English category names.
* Fill missing/untranslated categories with `'unknown'`.
* Retain `NaN` for missing physical dimensions (weight, length, etc.) to prevent statistical distortion.

In [5]:
# 1. Load Products and Translation tables
df_products = pd.read_sql("SELECT * FROM olist_products", con=engine)
df_translation = pd.read_sql("SELECT * FROM product_category_translation", con=engine)

# 2. Merge English category translations
df_products_clean = df_products.merge(
    df_translation, on="product_category_name", how="left"
)

# 3. Handle untranslated/missing categories
df_products_clean["product_category_name_english"] = df_products_clean[
    "product_category_name_english"
].fillna("unknown")

# 4. Verification Printout
print(f"✅ Products Cleaned! ({len(df_products_clean):,} rows)")
print(f"• Null English Categories Remaining: {df_products_clean['product_category_name_english'].isnull().sum()}")
print(f"• Sample English Categories: {df_products_clean['product_category_name_english'].head(3).tolist()}")

✅ Products Cleaned! (32,951 rows)
• Null English Categories Remaining: 0
• Sample English Categories: ['perfumery', 'auto', 'bed_bath_table']


### 2.4 Clean Order Items (`olist_order_items`)
* Convert `shipping_limit_date` to proper Pandas `datetime` format.

In [6]:
# 1. Load Order Items table
df_order_items = pd.read_sql("SELECT * FROM olist_order_items", con=engine)

# 2. Convert timestamp
df_order_items["shipping_limit_date"] = pd.to_datetime(df_order_items["shipping_limit_date"])

# 3. Verification Printout
print(f"✅ Order Items Cleaned! ({len(df_order_items):,} rows)")
print(f"• Sample Datetime Type Check: {type(df_order_items['shipping_limit_date'].iloc[0])}")

✅ Order Items Cleaned! (112,650 rows)
• Sample Datetime Type Check: <class 'pandas._libs.tslibs.timestamps.Timestamp'>


### 2.5 Clean Payments (`olist_order_payments`)
* Strip whitespace and convert `payment_type` strings to lowercase for consistent grouping.

In [7]:
# 1. Load Payments table
df_payments = pd.read_sql("SELECT * FROM olist_order_payments", con=engine)

# 2. Standardize payment type formatting
df_payments["payment_type"] = df_payments["payment_type"].str.strip().str.lower()

# 3. Verification Printout
print(f"✅ Order Payments Cleaned! ({len(df_payments):,} rows)")
print(f"• Payment Types Identified: {df_payments['payment_type'].unique().tolist()}")

✅ Order Payments Cleaned! (103,886 rows)
• Payment Types Identified: ['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined']


### 2.6 Clean Reviews (`olist_order_reviews`)
* Convert review creation and answer dates to Pandas `datetime` objects.
* Replace `NaN` values in comment titles and comment messages with clean fallback strings (`'No Title'`, `'No Comment'`).

In [8]:
# 1. Load Reviews table
df_reviews = pd.read_sql("SELECT * FROM olist_order_reviews", con=engine)

# 2. Convert timestamps
df_reviews["review_creation_date"] = pd.to_datetime(df_reviews["review_creation_date"])
df_reviews["review_answer_timestamp"] = pd.to_datetime(df_reviews["review_answer_timestamp"])

# 3. Fill missing text comments
df_reviews["review_comment_title"] = df_reviews["review_comment_title"].fillna("No Title")
df_reviews["review_comment_message"] = df_reviews["review_comment_message"].fillna("No Comment")

# 4. Verification Printout
print(f"✅ Order Reviews Cleaned! ({len(df_reviews):,} rows)")
print(f"• Null Titles Remaining: {df_reviews['review_comment_title'].isnull().sum()}")
print(f"• Null Messages Remaining: {df_reviews['review_comment_message'].isnull().sum()}")

✅ Order Reviews Cleaned! (99,224 rows)
• Null Titles Remaining: 0
• Null Messages Remaining: 0


### 2.7 Clean Sellers (`olist_sellers`)
* Strip leading/trailing whitespace from seller location fields.
* Convert `seller_city` to Title Case and `seller_state` to Uppercase.

In [9]:
# 1. Load Sellers table
df_sellers = pd.read_sql("SELECT * FROM olist_sellers", con=engine)

# 2. Clean location formatting
df_sellers["seller_city"] = df_sellers["seller_city"].str.strip().str.title()
df_sellers["seller_state"] = df_sellers["seller_state"].str.strip().str.upper()

# 3. Verification Printout
print(f"✅ Sellers Cleaned! ({len(df_sellers):,} rows)")
print(f"• Sample City Output: {df_sellers['seller_city'].head(3).tolist()}")
print(f"• Sample State Output: {df_sellers['seller_state'].head(3).tolist()}")

✅ Sellers Cleaned! (3,095 rows)
• Sample City Output: ['Santo Andre', 'Cariacica', 'Sao Goncalo']
• Sample State Output: ['SP', 'ES', 'RJ']


### 2.8 Clean Geolocation (`olist_geolocation`)
* Clean city names to Title Case and state codes to Uppercase.
* **Note on Duplicates:** The raw table contains multiple lat/lng observations per zip code prefix. These duplicate zip records are retained intentionally to preserve geospatial density.

In [10]:
# 1. Load Geolocation table
df_geolocation = pd.read_sql("SELECT * FROM olist_geolocation", con=engine)

# 2. Clean location formatting
df_geolocation["geolocation_city"] = df_geolocation["geolocation_city"].str.strip().str.title()
df_geolocation["geolocation_state"] = df_geolocation["geolocation_state"].str.strip().str.upper()

# 3. Verification Printout
print(f"✅ Geolocation Cleaned! ({len(df_geolocation):,} rows)")
print(f"• Sample City Output: {df_geolocation['geolocation_city'].head(3).tolist()}")

✅ Geolocation Cleaned! (1,000,163 rows)
• Sample City Output: ['Sao Paulo', 'Sao Paulo', 'Sao Paulo']


### 2.9 Clean Category Translation (`product_category_translation`)
* Strip whitespace and ensure lowercased category text for consistent lookup matching.

In [11]:
# 1. Load Translation table
df_translation_clean = pd.read_sql("SELECT * FROM product_category_translation", con=engine)

# 2. Standardize text
df_translation_clean["product_category_name"] = (
    df_translation_clean["product_category_name"].str.strip().str.lower()
)

# 3. Verification Printout
print(f"✅ Translation Lookup Cleaned! ({len(df_translation_clean):,} rows)")

✅ Translation Lookup Cleaned! (71 rows)


## 3. Save Cleaned Datasets to MySQL (`_clean` suffix)
Export all 9 cleaned DataFrames back into MySQL to establish our **Silver Data Layer**.

In [12]:
clean_tables = {
    "olist_customers_clean": df_customers,
    "olist_sellers_clean": df_sellers,
    "olist_products_clean": df_products_clean,
    "olist_orders_clean": df_orders,
    "olist_order_items_clean": df_order_items,
    "olist_order_payments_clean": df_payments,
    "olist_order_reviews_clean": df_reviews,
    "olist_geolocation_clean": df_geolocation,
    "product_category_translation_clean": df_translation_clean,
}

for table_name, df in clean_tables.items():
    df.to_sql(name=table_name, con=engine, if_exists="replace", index=False)
    print(f"✅ Exported: '{table_name}' ({len(df):,} rows)")

print("\n🎉 All 9 clean tables successfully exported to MySQL!")

✅ Exported: 'olist_customers_clean' (99,441 rows)
✅ Exported: 'olist_sellers_clean' (3,095 rows)
✅ Exported: 'olist_products_clean' (32,951 rows)
✅ Exported: 'olist_orders_clean' (99,441 rows)
✅ Exported: 'olist_order_items_clean' (112,650 rows)
✅ Exported: 'olist_order_payments_clean' (103,886 rows)
✅ Exported: 'olist_order_reviews_clean' (99,224 rows)
✅ Exported: 'olist_geolocation_clean' (1,000,163 rows)
✅ Exported: 'product_category_translation_clean' (71 rows)

🎉 All 9 clean tables successfully exported to MySQL!


In [13]:
import os

# Define output folder path
cleaned_dir = os.path.join("..", "data", "cleaned")
os.makedirs(cleaned_dir, exist_ok=True)

clean_tables = {
    "olist_customers_clean": df_customers,
    "olist_sellers_clean": df_sellers,
    "olist_products_clean": df_products_clean,
    "olist_orders_clean": df_orders,
    "olist_order_items_clean": df_order_items,
    "olist_order_payments_clean": df_payments,
    "olist_order_reviews_clean": df_reviews,
    "olist_geolocation_clean": df_geolocation,
    "product_category_translation_clean": df_translation_clean,
}

# Save to both MySQL and local CSV folder
for table_name, df in clean_tables.items():
    # Save to MySQL
    df.to_sql(name=table_name, con=engine, if_exists="replace", index=False)
    
    # Save to CSV in data/cleaned folder
    csv_path = os.path.join(cleaned_dir, f"{table_name}.csv")
    df.to_csv(csv_path, index=False)
    
    print(f"✅ Exported to MySQL & CSV: '{table_name}.csv' ({len(df):,} rows)")

print("\n🎉 All 9 clean tables exported to MySQL and saved in 'data/cleaned/' folder!")

✅ Exported to MySQL & CSV: 'olist_customers_clean.csv' (99,441 rows)
✅ Exported to MySQL & CSV: 'olist_sellers_clean.csv' (3,095 rows)
✅ Exported to MySQL & CSV: 'olist_products_clean.csv' (32,951 rows)
✅ Exported to MySQL & CSV: 'olist_orders_clean.csv' (99,441 rows)
✅ Exported to MySQL & CSV: 'olist_order_items_clean.csv' (112,650 rows)
✅ Exported to MySQL & CSV: 'olist_order_payments_clean.csv' (103,886 rows)
✅ Exported to MySQL & CSV: 'olist_order_reviews_clean.csv' (99,224 rows)
✅ Exported to MySQL & CSV: 'olist_geolocation_clean.csv' (1,000,163 rows)
✅ Exported to MySQL & CSV: 'product_category_translation_clean.csv' (71 rows)

🎉 All 9 clean tables exported to MySQL and saved in 'data/cleaned/' folder!


## 4. Cleaning Summary

* **Timestamps:** Standardized date columns across `orders`, `order_items`, and `reviews` to `datetime`.
* **Missing Translations:** Merged category translations into `products` and defaulted missing entries to `unknown`.
* **Missing Review Text:** Filled `NaN` comment titles/messages with `No Title` and `No Comment`.
* **Text Standardizing:** Trimmed whitespace and capitalized city/state names in `customers`, `sellers`, and `geolocation`.
* **Database Export:** Saved all 9 cleaned tables to MySQL with the `_clean` suffix.